In [2]:
import sys
import pprint
import networkx as nx
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parents[1]))

from src.graphs import ErdosRenyiGraph, ScaleFreeGraph, WattsStrogatzGraph, SNAPGraph, MultiLayerGraph
from src.messages import Layer
from src.visualization.plots import analyse_graph
from src.visualization.metrics import compute_graph_metrics, compute_multilayer_metrics, metrics_to_json_safe

In [3]:
def _fmt(m: dict) -> dict:
    return {k: v for k, v in m.items() if not k.startswith("_")}

def show_full(G, name: str, seed: int = 42) -> dict:
    m = metrics_to_json_safe(compute_graph_metrics(G, seed=seed))
    print(f"\n=== {name} ===")
    pprint.pprint(_fmt(m))
    return m

def show_multilayer(ml_G, name: str, seed: int = 42) -> dict:
    analog_edges = [(u, v) for u, v, d in ml_G.edges(data=True) if d.get("layer") == Layer.ANALOG]
    digital_edges = [(u, v) for u, v, d in ml_G.edges(data=True) if d.get("layer") == Layer.DIGITAL]

    G_a: nx.Graph = nx.Graph()
    G_a.add_nodes_from(ml_G.nodes())
    G_a.add_edges_from(analog_edges)

    G_d: nx.DiGraph = nx.DiGraph()
    G_d.add_nodes_from(ml_G.nodes())
    G_d.add_edges_from(digital_edges)

    m_a = metrics_to_json_safe(compute_graph_metrics(G_a, seed=seed))
    m_d = metrics_to_json_safe(compute_graph_metrics(G_d, seed=seed))
    m_ml = metrics_to_json_safe(compute_multilayer_metrics(ml_G, G_a, G_d))

    print(f"\n=== {name} — Analógica ===");  pprint.pprint(_fmt(m_a))
    print(f"\n=== {name} — Digital ===");    pprint.pprint(_fmt(m_d))
    print(f"\n=== {name} — Multicapa ===");  pprint.pprint(_fmt(m_ml))
    return {"analog": m_a, "digital": m_d, "multilayer": m_ml}

## Erdős-Rényi

In [4]:
# ── Parametros ──────────────────────────────────────────────
NUM_NODES = 10_000
EDGE_PROB = 0.005
SEED      = 42
# ────────────────────────────────────────────────────────────

g = ErdosRenyiGraph(num_nodes=NUM_NODES, edge_prob=EDGE_PROB, seed=SEED)
show_full(g.graph, f'Erdos-Renyi  G({NUM_NODES}, p={EDGE_PROB})', seed=SEED)


=== Erdos-Renyi  G(10000, p=0.005) ===
{'assortativity': -0.0005486341134572406,
 'avg_shortest_path': 2.7741860852751943,
 'betweenness_avg': 0.00017706681619424148,
 'betweenness_max': 0.0005217391829132543,
 'density': 0.005000740074007401,
 'diameter': 4,
 'in_degree_avg': 50.0024,
 'in_degree_max': 77,
 'lcc_fraction': 1.0,
 'lcc_size': 10000,
 'm': 500024,
 'n': 10000,
 'n_components': 1,
 'n_scc': 1,
 'out_degree_avg': 50.0024,
 'out_degree_max': 77,
 'pagerank_avg': 9.999999999999999e-05,
 'pagerank_max': 0.00014581386160561979,
 'powerlaw_gamma_in': 1.2176254376228495,
 'powerlaw_gamma_out': 1.2176254376228495,
 'reciprocity': 1.0,
 'transitivity': 0.0050035841582922996}


{'n': 10000,
 'm': 500024,
 'density': 0.005000740074007401,
 'n_components': 1,
 'lcc_size': 10000,
 'lcc_fraction': 1.0,
 'transitivity': 0.0050035841582922996,
 'avg_shortest_path': 2.7741860852751943,
 'diameter': 4,
 'assortativity': -0.0005486341134572406,
 'betweenness_max': 0.0005217391829132543,
 'betweenness_avg': 0.00017706681619424148,
 'pagerank_max': 0.00014581386160561979,
 'pagerank_avg': 9.999999999999999e-05,
 'n_scc': 1,
 'reciprocity': 1.0,
 'in_degree_max': 77,
 'in_degree_avg': 50.0024,
 'out_degree_max': 77,
 'out_degree_avg': 50.0024,
 'powerlaw_gamma_in': 1.2176254376228495,
 'powerlaw_gamma_out': 1.2176254376228495}

## Watts-Strogatz (Small World)

In [5]:
# ── Parametros ──────────────────────────────────────────────
NUM_NODES   = 10_000
K           = 150
REWIRE_PROB = 0.02
SEED        = 42
# ────────────────────────────────────────────────────────────

g = WattsStrogatzGraph(num_nodes=NUM_NODES, k=K, rewire_prob=REWIRE_PROB, seed=SEED)
show_full(g.graph, f'Watts-Strogatz  n={NUM_NODES}  k={K}  p={REWIRE_PROB}', seed=SEED)


=== Watts-Strogatz  n=10000  k=150  p=0.02 ===
{'assortativity': 0.00016626786289248424,
 'avg_shortest_path': 2.8744334433443344,
 'betweenness_avg': 0.00018722367942862564,
 'betweenness_max': 0.0007459445686982052,
 'density': 0.015001500150015001,
 'diameter': 4,
 'in_degree_avg': 150.0,
 'in_degree_max': 159,
 'lcc_fraction': 1.0,
 'lcc_size': 10000,
 'm': 1500000,
 'n': 10000,
 'n_components': 1,
 'n_scc': 1,
 'out_degree_avg': 150.0,
 'out_degree_max': 159,
 'pagerank_avg': 0.00010000000000000002,
 'pagerank_max': 0.00010503556703783007,
 'powerlaw_gamma_in': 1.175324276885645,
 'powerlaw_gamma_out': 1.175324276885645,
 'reciprocity': 1.0,
 'transitivity': 0.7011765778512933}


{'n': 10000,
 'm': 1500000,
 'density': 0.015001500150015001,
 'n_components': 1,
 'lcc_size': 10000,
 'lcc_fraction': 1.0,
 'transitivity': 0.7011765778512933,
 'avg_shortest_path': 2.8744334433443344,
 'diameter': 4,
 'assortativity': 0.00016626786289248424,
 'betweenness_max': 0.0007459445686982052,
 'betweenness_avg': 0.00018722367942862564,
 'pagerank_max': 0.00010503556703783007,
 'pagerank_avg': 0.00010000000000000002,
 'n_scc': 1,
 'reciprocity': 1.0,
 'in_degree_max': 159,
 'in_degree_avg': 150.0,
 'out_degree_max': 159,
 'out_degree_avg': 150.0,
 'powerlaw_gamma_in': 1.175324276885645,
 'powerlaw_gamma_out': 1.175324276885645}

## Scale-Free (Barabási-Albert extendido)

In [ ]:
# ── Parametros ──────────────────────────────────────────────
NUM_NODES  = 10_000
ALPHA      = 0.41
BETA       = 0.54
GAMMA      = 0.05
DELTA_IN   = 0.2
DELTA_OUT  = 0.0
SEED       = 42
# ────────────────────────────────────────────────────────────

g = ScaleFreeGraph(
    num_nodes=NUM_NODES, alpha=ALPHA, beta=BETA, gamma=GAMMA,
    delta_in=DELTA_IN, delta_out=DELTA_OUT, seed=SEED,
)
show_full(
    g.graph,
    f'Scale-Free  n={NUM_NODES}  a={ALPHA}  b={BETA}  g={GAMMA}',
    seed=SEED,
)


=== Scale-Free  n=10000  a=0.41  b=0.54  g=0.05 ===
{'assortativity': -0.15366430116956464,
 'avg_shortest_path': 4.321214837086935,
 'betweenness_avg': 1.6217727017980276e-05,
 'betweenness_max': 0.03726600186291941,
 'density': 0.0001869086908690869,
 'diameter': 14,
 'in_degree_avg': 1.8689,
 'in_degree_max': 2836,
 'lcc_fraction': 1.0,
 'lcc_size': 10000,
 'm': 18689,
 'n': 10000,
 'n_components': 1,
 'n_scc': 9693,
 'out_degree_avg': 1.8689,
 'out_degree_max': 139,
 'pagerank_avg': 0.00010000000000000005,
 'pagerank_max': 0.08873832476367925,
 'powerlaw_gamma_in': 1.685536109570753,
 'powerlaw_gamma_out': 1.9005412055525923,
 'reciprocity': 0.005243726256086468,
 'transitivity': 0.07818978273290651}


{'n': 10000,
 'm': 18689,
 'density': 0.0001869086908690869,
 'n_components': 1,
 'lcc_size': 10000,
 'lcc_fraction': 1.0,
 'transitivity': 0.07818978273290651,
 'avg_shortest_path': 4.321214837086935,
 'diameter': 14,
 'assortativity': -0.15366430116956464,
 'betweenness_max': 0.03726600186291941,
 'betweenness_avg': 1.6217727017980276e-05,
 'pagerank_max': 0.08873832476367925,
 'pagerank_avg': 0.00010000000000000005,
 'n_scc': 9693,
 'reciprocity': 0.005243726256086468,
 'in_degree_max': 2836,
 'in_degree_avg': 1.8689,
 'out_degree_max': 139,
 'out_degree_avg': 1.8689,
 'powerlaw_gamma_in': 1.685536109570753,
 'powerlaw_gamma_out': 1.9005412055525923}

## SNAP — Red real

In [7]:
# ── Parametros ──────────────────────────────────────────────
DATASET   = 'ego-Facebook'
CACHE_DIR = '../../data/snap'
DIRECTED  = False
SEED      = 42
# ────────────────────────────────────────────────────────────

g = SNAPGraph(dataset_name=DATASET, cache_dir=CACHE_DIR, directed=DIRECTED, seed=SEED)
tipo = 'dirigido' if DIRECTED else 'no dirigido'
show_full(g.graph, f'SNAP — {DATASET} ({tipo})', seed=SEED)

[cache] ego-Facebook: ya existe en ../../data/snap/raw/ego-Facebook.txt.gz
[cache] ego-Facebook: ya descomprimido en ../../data/snap/processed/ego-Facebook.txt

=== SNAP — ego-Facebook (no dirigido) ===
{'assortativity': 0.0635772291856494,
 'avg_shortest_path': 3.626643552913984,
 'betweenness_avg': 0.0006711510308511188,
 'betweenness_max': 0.48304504473076193,
 'density': 0.010819963503439287,
 'diameter': 8,
 'in_degree_avg': 43.69101262688784,
 'in_degree_max': 1045,
 'lcc_fraction': 1.0,
 'lcc_size': 4039,
 'm': 176468,
 'n': 4039,
 'n_components': 1,
 'n_scc': 1,
 'out_degree_avg': 43.69101262688784,
 'out_degree_max': 1045,
 'pagerank_avg': 0.0002475860361475613,
 'pagerank_max': 0.0076145868447496,
 'powerlaw_gamma_in': 1.2587730752094903,
 'powerlaw_gamma_out': 1.2587730752094903,
 'reciprocity': 1.0,
 'transitivity': 0.5191742775433075}


{'n': 4039,
 'm': 176468,
 'density': 0.010819963503439287,
 'n_components': 1,
 'lcc_size': 4039,
 'lcc_fraction': 1.0,
 'transitivity': 0.5191742775433075,
 'avg_shortest_path': 3.626643552913984,
 'diameter': 8,
 'assortativity': 0.0635772291856494,
 'betweenness_max': 0.48304504473076193,
 'betweenness_avg': 0.0006711510308511188,
 'pagerank_max': 0.0076145868447496,
 'pagerank_avg': 0.0002475860361475613,
 'n_scc': 1,
 'reciprocity': 1.0,
 'in_degree_max': 1045,
 'in_degree_avg': 43.69101262688784,
 'out_degree_max': 1045,
 'out_degree_avg': 43.69101262688784,
 'powerlaw_gamma_in': 1.2587730752094903,
 'powerlaw_gamma_out': 1.2587730752094903}